In [ ]:
!pip install -U sentence-transformers transformers faiss-cpu sentencepiece --quiet

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
documents = [
    "Paris is the capital of France.",
    "Berlin is the capital of Germany.",
    "Python is a programming language."
]

In [ ]:
import faiss
import numpy as np

doc_embeddings = embed_model.encode(documents)

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

In [ ]:
query = "What is the capital of France?"

query_embedding = embed_model.encode([query])

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(query_embedding, doc_embeddings)[0]

sorted_scores = sorted(scores, reverse=True)

confidence = sorted_scores[0] - sorted_scores[1]

best_index = scores.argmax()
context = documents[best_index]

print("CONFIDENCE SCORE:", confidence)
print("RETRIEVED CONTEXT:", context)

CONFIDENCE SCORE: 0.45270973
RETRIEVED CONTEXT: Paris is the capital of France.


In [9]:
threshold = 0.1  # you can experiment

if confidence < threshold:
    print("\nLOW CONFIDENCE → I don't know")
else:
    prompt = f"""
    Use the following context to answer the question.

    Context: {context}

    Question: {query}
    """

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(**inputs, max_new_tokens=30)

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\nFINAL ANSWER:")
    print(answer)


FINAL ANSWER:
Paris
